# 🎬 YT Short Clipper Pro — Colab

Run everything in 3 cells. Just click ▶️ and wait for the Ngrok URL.

**Features**: AI Analysis, Face Tracking, Karaoke Subtitle, B-Roll Overlay, Background Music

**Required**: Cookies file (Netscape format) dari browser yang login YouTube.

## 1. Install & Upload Cookies

In [ ]:
#@title 1. Install YT-Short-Clipper + Upload Cookies { display-mode: "form" }
#@markdown Clone repo, install deps, and upload cookies.txt for YouTube download.

import subprocess, sys, os
from google.colab import files

print("="*60)
print("📦 Installing YT-Short-Clipper-Offline...")
print("="*60)

# --- Clone or pull repo ---
repo_url = "https://github.com/Chukie99/yt-short-clipper-offline.git"
repo_dir = "/content/yt-short-clipper-offline"

if os.path.exists(repo_dir):
    print("\n[1/6] Repository exists, pulling latest...")
    subprocess.run(["git", "-C", repo_dir, "pull"], capture_output=True)
else:
    print("\n[1/6] Cloning repository...")
    subprocess.run(["git", "clone", repo_url, repo_dir], capture_output=True)

os.chdir(repo_dir)
sys.path.insert(0, repo_dir)
print("  ✅ Repository ready!")

# --- Install system dependencies ---
print("\n[2/6] Installing system dependencies (ffmpeg + nodejs)...")
subprocess.run(["apt-get", "-qq", "install", "ffmpeg"], capture_output=True)
subprocess.run(["apt-get", "-qq", "install", "nodejs"], capture_output=True)
subprocess.run(["apt-get", "-qq", "install", "npm"], capture_output=True)
print("  ✅ FFmpeg + Node.js installed!")

# --- Upgrade yt-dlp to latest ---
print("\n[3/6] Upgrading yt-dlp to latest version...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "yt-dlp[default]"], capture_output=True)
import yt_dlp
print(f"  ✅ yt-dlp version: {yt_dlp.version.__version__}")

# --- Install Python dependencies ---
print("\n[4/6] Installing Python dependencies...")
deps = [
    "opencv-python-headless", "numpy", "Pillow",
    "requests", "mediapipe", "python-dotenv", "faster-whisper",
    "google-genai", "streamlit", "pyngrok"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, capture_output=True)
print("  ✅ Python packages installed!")

# --- Upload cookies.txt ---
print("\n[5/6] Upload cookies.txt")
print("\n📋 CARA EXPORT COOKIES:")
print("1. Install extension 'Get cookies.txt LOCALLY' di Chrome")
print("2. Buka YouTube.com, pastikan sudah LOGIN")
print("3. Klik icon extension → Export (Netscape format)")
print("4. Upload file .txt hasil export di bawah ini")
print("\n" + "-"*40)

cookies_path = "/content/yt-short-clipper-offline/cookies.txt"
# Remove old cookies to force fresh upload
if os.path.exists(cookies_path):
    os.remove(cookies_path)

print("⏳ Klik tombol 'Choose Files' di bawah ini...")
uploaded = files.upload()
if uploaded:
    filename = list(uploaded.keys())[0]
    with open(cookies_path, "wb") as f:
        f.write(uploaded[filename])
    print(f"\n✅ Cookies saved: {cookies_path}")
else:
    print("\n⚠️ Tidak ada file di-upload!")
    print("Tanpa cookies, download akan gagal karena YouTube bot detection.")
    print("Upload cookies.txt dulu, lalu jalankan cell ini lagi.")

# --- Verify Node.js ---
print("\n[6/6] Verifying installation...")
node_check = subprocess.run(["node", "--version"], capture_output=True, text=True)
if node_check.returncode == 0:
    print(f"  ✅ Node.js: {node_check.stdout.strip()}")
else:
    print("  ⚠️ Node.js not found - yt-dlp n-challenge may fail")

for f in ["clipper_core.py", "app.py", "cookies.txt"]:
    status = "✅" if os.path.exists(os.path.join(repo_dir, f)) else "❌"
    print(f"  {status} {f}")

print("\n" + "="*60)
print("YT-Short-Clipper is ready!")
print("="*60)

## 2. Configure & Launch WebUI

In [ ]:
#@title 2. Configure Colab Tunnel & Launch WebUI { display-mode: "form" }
#@markdown Configure API keys and launch Streamlit with Ngrok tunnel.

import os, sys, time, subprocess
from pathlib import Path

# --- Colab Form Parameters ---
ngrok_authtoken = "" #@param {type:"string"}
ai_provider = "Gemini" #@param ["Gemini", "Groq", "OpenRouter"]
api_key = "" #@param {type:"string"}

# --- Change to repo directory ---
repo_dir = "/content/yt-short-clipper-offline"
os.chdir(repo_dir)
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# --- Setup directories ---
from clipper_core import setup_directories
setup_directories(
    temp_dir="/content/temp",
    output_dir="/content/drive/MyDrive/YTShortClipper/output",
    config_file="/content/drive/MyDrive/YTShortClipper/config.json",
)

# --- Save cookies path to config ---
cookies_file = "/content/yt-short-clipper-offline/cookies.txt"
if os.path.exists(cookies_file):
    print(f"✅ Cookies found: {cookies_file}")
else:
    print("⚠️ Cookies not found. Download will likely fail.")

# --- Export config to .env ---
config = {
    "ai_provider": ai_provider,
    "gemini_api_key": api_key if ai_provider == "Gemini" else "",
    "groq_api_key": api_key if ai_provider == "Groq" else "",
    "openrouter_api_key": api_key if ai_provider == "OpenRouter" else "",
    "ngrok_authtoken": ngrok_authtoken,
    "cookies_path": cookies_file,
}

env_path = Path(repo_dir) / ".env"
with open(env_path, "w") as f:
    for key, val in config.items():
        f.write(f"{key.upper()}={val}\n")

# Also save to config.json for Streamlit
import json
config_json_path = Path(repo_dir) / "config.json"
cfg_data = {}
if config_json_path.exists():
    with open(config_json_path, "r") as f:
        cfg_data = json.load(f)
cfg_data["cookies_path"] = cookies_file
cfg_data["ai_provider"] = ai_provider
cfg_data["gemini_api_key"] = config["gemini_api_key"]
cfg_data["groq_api_key"] = config["groq_api_key"]
cfg_data["openrouter_api_key"] = config["openrouter_api_key"]
with open(config_json_path, "w") as f:
    json.dump(cfg_data, f, indent=2)

print("✅ Config exported")

# --- Initialize Ngrok ---
public_url = None
if ngrok_authtoken:
    try:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = ngrok_authtoken
        ngrok.kill()
        tunnel = ngrok.connect(8501, "http")
        public_url = tunnel.public_url
        print(f"✅ Ngrok tunnel created: {public_url}")
    except Exception as e:
        print(f"⚠️ Ngrok error: {e}")
else:
    print("⚠️ No Ngrok token. Using Colab direct link.")

# --- Launch Streamlit ---
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.address=0.0.0.0",
     "--browser.gatherUsageStats=false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("\n⏳ Waiting for Streamlit to start...")
for i in range(20):
    time.sleep(1)
    if process.poll() is not None:
        print(f"❌ Streamlit crashed! Exit code: {process.returncode}")
        break
else:
    print("✅ Streamlit is running!")

# --- Show public URL ---
print("\n" + "="*60)
if public_url:
    print(f"YT-Short-Clipper-Offline is ready:")
    print(public_url)
else:
    print("Colab Direct Link:")
    print("http://localhost:8501")
print("="*60)
print("\nOpen the URL above in your browser to access the WebUI!")
print("To stop: run 'pkill -f streamlit' in a new cell\n")